# CSE 151B Competition Prompt Engineering

In this notebook we try different prompt engineering techniques on a small subset of the data

## 1. UV Environment Setup (I don't use uv because I'm on Colab)

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

downloading uv 0.11.8 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment with seed packages at: .venv
 + pip==26.1
Activate with: source .venv/bin/activate
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 17.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 11.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 44.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 25.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 59.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 58.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 64.8 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 70.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 294.9/295.2 MB 39.0 MB/s eta 0:00:01

In [ ]:
# # activate venv after installation. This needs to be run everytime.
# !source ./.venv/bin/activate

/bin/bash: line 1: ./.venv/bin/activate: No such file or directory


## Colab setup

### Mount Google Drive
Connect to Google Drive so you can load the dataset and save your results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Run the cell below every time to install necessary libraries into Colab. There's no ipkernel or anything so I don't bother with uv.

In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers protobuf tensorflow tensorflow-cpu

!pip install -q \
  torch==2.6.0 \
  torchvision==0.21.0 \
  torchaudio==2.6.0 \
  --index-url https://download.pytorch.org/whl/cu124

!pip install -q \
  vllm==0.8.5 \
  transformers==4.51.3 \
  accelerate \
  bitsandbytes \
  tqdm \
  sympy \
  antlr4-python3-runtime==4.11.1 \
  protobuf==4.25.3

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 102.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

print("ALL IMPORTS OK")

torch: 2.6.0+cu124
cuda: 12.4
gpu: NVIDIA A100-SXM4-80GB
INFO 05-11 14:09:34 [__init__.py:239] Automatically detected platform cuda.
ALL IMPORTS OK


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Please connect to a GPU runtime (Runtime > Change runtime type).")

NVIDIA A100-SXM4-80GB


In [ ]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES (Changed from 1 to 0 for Colab)
DATA_PATH   = "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition/data/public.jsonl"
OUTPUT_PATH = "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition/results/prompt_engineering.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

## Prompt Engineering

To test multiple prompts, I created a dictionary of prompts and edited build_prompt() to take in an additional argument for which prompt it should use.

"baseline" - prompts given in the starter notebook

"strict_boxed" - Tries to make sure the answer is in exactly the format required for the judger, by asking not to put any text after the boxed answer and asking for exactly one boxed letter for the MCQs.

"concise" - Ask the model to think more concisely. This should hopefully keep it from thinking too long and might make the model faster. We are curious on whether this will result in an decrease in accuracy

"self_check" - Asking the model to double-check their answers, and for the MCQs, asking it to evaluate all other possible answers.

In [ ]:
PROMPT_VARIANTS = {
    "baseline": {
        "math": (
            "You are an expert mathematician. Solve the problem step-by-step. "
            "Put your final answer inside \\boxed{}. "
            "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
            "e.g. \\boxed{3, 7}."
        ),
        "mcq": (
            "You are an expert mathematician. "
            "Read the problem and the answer choices below, then select the single best answer. "
            "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },

    "strict_boxed": {
        "math": (
            "You are an expert mathematician. Solve the problem carefully. "
            "Your final answer must appear exactly once, inside \\boxed{}. "
            "Do not put any text after the boxed answer. "
            "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
            "e.g. \\boxed{3, 7}."
        ),
        "mcq": (
            "You are an expert mathematician. Solve the multiple-choice problem carefully. "
            "Your final response must be exactly ONE boxed letter: \\boxed{A}, \\boxed{B}, \\boxed{C}, \\boxed{D}, or \\boxed{E}. "
            "Do not include any text after the boxed answer."
        ),
    },

    "concise": {
        "math": (
            "You are an expert mathematician. Use concise step-by-step reasoning. "
            "Avoid unnecessary explanation. "
            "Put your final answer inside \\boxed{}. "
            "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
            "e.g. \\boxed{3, 7}."
        ),
        "mcq": (
            "You are an expert mathematician. Solve the problem concisely."
            "Choose the single best answer choice. "
            "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },

    "self_check": {
        "math": (
            "You are an expert mathematician. Solve the problem step-by-step. "
            "Before giving the final answer, perform a brief sanity check."
            "Put your final answer inside \\boxed{}. "
            "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
            "e.g. \\boxed{3, 7}."
        ),
        "mcq": (
            "You are an expert mathematician. "
            "Read the problem and the answer choices below, then select the single best answer. "
            "After arriving at a single answer, double check your reasoning against all the other options."
            "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
        ),
    }
}

In [ ]:
PROMPT_VARIANTS = {
    "concise": {
        "math": (
            "You are an expert mathematician. Use concise step-by-step reasoning. "
            "Avoid unnecessary explanation. "
            "Put your final answer inside \\boxed{}. "
            "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
            "e.g. \\boxed{3, 7}."
        ),
        "mcq": (
            "You are an expert mathematician. Solve the problem concisely."
            "Choose the single best answer choice. "
            "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },
    "concise_verify": {
        "math": (
            "You are an expert mathematician. Solve the problem with concise step-by-step reasoning. "
            "Before finalizing, briefly check for arithmetic or logic mistakes. "
            "Put the final answer inside \\boxed{}. "
            "If there are multiple sub-answers, separate them by commas inside a single \\boxed{}, "
            "e.g. \\boxed{3, 7}."
        ),
        "mcq": (
            "You are an expert mathematician. Solve the multiple-choice problem concisely. "
            "Eliminate clearly wrong choices when useful, then choose the single best answer. "
            "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },

    "minimal_reasoning": {
        "math": (
            "Solve the math problem. Show only the necessary reasoning. "
            "End with the final answer inside \\boxed{}. "
            "If there are multiple answers, put them comma-separated inside one box, "
            "e.g. \\boxed{3, 7}."
        ),
        "mcq": (
            "Solve the multiple-choice math problem. "
            "Output ONLY the correct choice letter inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },
    "answer_focused": {
        "math": (
            "Find the answer to the problem. Reason efficiently and avoid extra commentary. "
            "The final answer must be inside \\boxed{}. "
            "For multiple sub-answers, use one boxed comma-separated answer, "
            "e.g. \\boxed{3, 7}."
        ),
        "mcq": (
            "Find the correct answer choice. "
            "Do not explain. "
            "Output ONLY one boxed letter: \\boxed{A}, \\boxed{B}, \\boxed{C}, \\boxed{D}, or \\boxed{E}."
        ),
    },
    "competition_style": {
        "math": (
            "Solve the problem as in a math competition. "
            "Use short, accurate reasoning. "
            "Put the final answer inside \\boxed{}. "
            "If multiple answers are required, separate them by commas inside one box, "
            "e.g. \\boxed{3, 7}."
        ),
        "mcq": (
            "Solve the multiple-choice problem as in a math competition. "
            "Choose the single best option. "
            "Output ONLY the letter inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },
}

Updated build_prompt

In [ ]:
def build_prompt(
    question: str,
    options: Optional[list],
    variant: str = "baseline"
    )-> tuple[str, str]:

    """Return (system_prompt, user_prompt) for a question."""

    prompts = PROMPT_VARIANTS[variant]

    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return prompts['mcq'], f"{question}\n\nOptions:\n{opts_text}"
    return prompts['math'], question

In [ ]:
# Verify with samples

variant = 'answer_focused'

for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"), variant)
    print(f"── {label} system prompt (first 200 chars) ──")
    print(sys_p[:200], "...\n")
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ system prompt (first 200 chars) ──
Find the correct answer choice. Do not explain. Output ONLY one boxed letter: \boxed{A}, \boxed{B}, \boxed{C}, \boxed{D}, or \boxed{E}. ...

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form system prompt (first 200 chars) ──
Find the answer to the problem. Reason efficiently and avoid extra commentary. The final answer must be inside \boxed{}. For multiple sub-answers, use one boxed comma-separated answer, e.g. \boxed{3,  ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [25]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

INFO 05-11 15:48:40 [config.py:717] This model supports multiple tasks: {'reward', 'embed', 'classify', 'score', 'generate'}. Defaulting to 'generate'.
WARNING 05-11 15:48:40 [config.py:830] bitsandbytes quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 05-11 15:48:40 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 05-11 15:50:24 [core_client.py:439] Core engine process 0 ready.
Model loaded.


### 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = 'left'

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

ERROR:bitsandbytes.cextension:bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.


### Generate with vLLM

First, I create a dictionary "prompts" where keys are the type of prompt and values are a list of prompts. Each different list has the same 20 corresponding questions, just different system prompts

In [26]:
# Build prompts for first N entries
N = 100
prompts = {}
for variant in PROMPT_VARIANTS:
  prompts[variant] = []
  for item in data[:N]:
    system, user = build_prompt(item["question"], item.get("options"), variant)
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts[variant].append(prompt_text)

I generate responses for all variants

In [27]:
responses = {}
finish_reasons = {}

for variant in PROMPT_VARIANTS:
    print(f"Generating responses for {len(prompts[variant])} questions for variant: {variant}...")

    outputs = llm.generate(prompts[variant], sampling_params=sampling_params)

    responses[variant] = [
        out.outputs[0].text.strip()
        for out in outputs
    ]

    finish_reasons[variant] = [
        out.outputs[0].finish_reason
        for out in outputs
    ]

Generating responses for 100 questions for variant: concise...


Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses for 100 questions for variant: concise_verify...


Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses for 100 questions for variant: minimal_reasoning...


Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses for 100 questions for variant: answer_focused...


Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses for 100 questions for variant: competition_style...


Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [28]:
# Preview first question for each variant
for variant in PROMPT_VARIANTS:
#for i in range(min(3, len(responses))):
    i = 10
    print(f"\n── Response {i} (id={data[i].get('id')}), (variant: {variant}) ──")
    print(responses[variant][i])
    #print(responses[variant][i][:400], "..." if len(responses[variant][i]) > 400 else "")


── Response 10 (id=10), (variant: concise) ──
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so I need to compute the sum of the floor of log base 2 of each integer from 1 to 2018, and then find that sum mod 1000. Let's start by recalling what floor(log2 n) means for different n. For example, floor(log2 n) = t when n is in [2^t, 2^(t+1) - 1], right? Wait, let's check small numbers to make sure.

For n=1: log2 1 = 0, floor is 0. So when is floor(log2 n) = 0? When n is from 1 to 2^1 - 1 = 1, yes, only n=1.

floor(log2 n)=1 when n is from 2^1=2 to 2^2 -1=3, so n=2,3.

floor(log2 n)=2 when n=4 to 7, since 2^2=4, 2^3-1=7.

Wait, hold on, let's confirm n=4: log2 4=2, floor is 2, correct. n=7: log2 7 ≈ 2.807, floor is 2, correct. n=8: log2 8=3, floor is 3, so yes, floor(log2 n)=t for n in [2^t, 2^(t+1)-1].

But wait, n starts at 1 here. Let's list out the ranges for each t:

t=0: n where 2^0 ≤ n < 2^1, so

In [29]:
from collections import Counter

for variant in PROMPT_VARIANTS:
    print(f"\nVariant: {variant}")
    print(Counter(finish_reasons[variant]))


Variant: concise
Counter({'stop': 91, 'length': 9})

Variant: concise_verify
Counter({'stop': 94, 'length': 6})

Variant: minimal_reasoning
Counter({'stop': 92, 'length': 8})

Variant: answer_focused
Counter({'stop': 93, 'length': 7})

Variant: competition_style
Counter({'stop': 92, 'length': 8})


In [30]:
# Compare answer lengths between the four variants

response_lengths = {} # To store lengths for each variant

for variant in PROMPT_VARIANTS:
    lengths = [len(response) for response in responses[variant]]
    response_lengths[variant] = lengths

print("Average Response Lengths per Variant:")
print("=" * 50)
for variant, lengths in response_lengths.items():
    if lengths:
        average_length = sum(lengths) / len(lengths)
        print(f"  {variant.ljust(15)}: {average_length:.2f} characters")
    else:
        print(f"  {variant.ljust(15)}: No responses generated.")
print("=" * 50)

Average Response Lengths per Variant:
  concise        : 13487.62 characters
  concise_verify : 14539.01 characters
  minimal_reasoning: 14360.71 characters
  answer_focused : 12310.70 characters
  competition_style: 17031.16 characters


#### Generate with Transformers (for Datahub)

In [ ]:
# import torch

# print("CUDA available:", torch.cuda.is_available())

# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(0))
# else:
#     print("No GPU detected")

In [ ]:
# # Build prompts for first 5 entries
# MAX_TOKENS = 2048
# N = 5 # Number of questions

# prompts = []
# for item in data[:N]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=2048 #16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.0,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")



In [ ]:
print(prompts[0])
print(responses[0])

<|im_start|>system
You are an expert mathematician. Solve the problem step-by-step. Put your final answer inside \boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \boxed{}, e.g. \boxed{3, 7}.<|im_end|>
<|im_start|>user
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]<|im_end|>
<|im_start|>assistant
<think>

This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so I need to find the sum of the first 325 positive even whole numbers. Hmm, let's start by recalling what the first few positive even whole numbers are. Wait, positive even whole numbers start at 2, right? Like 2, 4, 6, 8, ..., so the nth one would be 2n? Let me check that. First one, n=1: 2*1=2, good. Second one, n=2: 4, yep, that works. So the kth positive even whole number is 2k for k from 1 to... well, in this problem, up to 325 terms? Wait, hold on, the problem says "first 325 posi

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [31]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
import sys
sys.path.insert(0, "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition")
from judger import Judger
judger = Judger(strict_extract=False)

results = {}
for variant in PROMPT_VARIANTS:
  results[variant] = []
  for item, response in tqdm(zip(data, responses[variant]), total=len(data), desc="Scoring"):
      is_mcq = bool(item.get("options"))
      gold   = item["answer"]

      if is_mcq:
          correct = score_mcq(response, str(gold))
      else:
          gold_list = gold if isinstance(gold, list) else [gold]
          try:
              correct = judger.auto_judge(
                  pred=response,
                  gold=gold_list,
                  options=[[]] * len(gold_list),
              )
          except Exception:
              correct = False

      results[variant].append({
          "id":       item.get("id"),
          "is_mcq":   is_mcq,
          "gold":     gold,
          "response": response,
          "correct":  correct,
      })

#print(f"Scoring complete. {len(results)} results.")
print("Scoring complete")



Scoring:   0%|          | 0/1126 [00:00<?, ?it/s]

Scoring:   0%|          | 1/1126 [00:00<02:51,  6.58it/s]

Scoring:   1%|          | 6/1126 [00:00<01:31, 12.28it/s]

Scoring:   1%|          | 8/1126 [00:00<01:25, 13.08it/s]

Scoring:   1%|▏         | 16/1126 [00:00<00:39, 28.31it/s]

Scoring:   2%|▏         | 20/1126 [00:01<00:52, 21.03it/s]

Scoring:   2%|▏         | 26/1126 [00:01<00:43, 25.23it/s]

Scoring:   3%|▎         | 30/1126 [00:01<01:24, 12.94it/s]

Scoring:   3%|▎         | 33/1126 [00:02<01:42, 10.63it/s]

Scoring:   3%|▎         | 35/1126 [00:02<01:59,  9.11it/s]

Scoring:   3%|▎         | 37/1126 [00:02<01:47, 10.15it/s]

Scoring:   4%|▎         | 40/1126 [00:02<01:27, 12.34it/s]

Scoring:   4%|▎         | 42/1126 [00:03<02:42,  6.67it/s]

Scoring:   4%|▍         | 47/1126 [00:03<01:40, 10.76it/s]

Scoring:   5%|▍         | 56/1126 [00:04<01:12, 14.76it/s]

Scoring:   6%|▌         | 65/1126 [00:04<00:51, 20.75it/s]

Scoring:   6%|▌         | 68/1126 [00:05<01:20, 13

Scoring complete


## 8. Summary

Print accuracy broken down by question type.

In [32]:
for variant in PROMPT_VARIANTS:
  print(f"Variant: {variant}")
  mcq_res  = [r for r in results[variant] if r["is_mcq"]]
  free_res = [r for r in results[variant] if not r["is_mcq"]]

  def acc(subset):
      return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

  print("=" * 50)
  print("EVALUATION RESULTS")
  print("=" * 50)
  print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
  print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
  print(f"  Overall    : {sum(r['correct'] for r in results[variant]):4d} / {len(results[variant]):4d}  ({acc(results[variant]):.2f}%)")
  print("=" * 50)

Variant: concise
EVALUATION RESULTS
  MCQ        :   26 /   38  (68.42%)
  Free-form  :   32 /   62  (51.61%)
  Overall    :   58 /  100  (58.00%)
Variant: concise_verify
EVALUATION RESULTS
  MCQ        :   28 /   38  (73.68%)
  Free-form  :   31 /   62  (50.00%)
  Overall    :   59 /  100  (59.00%)
Variant: minimal_reasoning
EVALUATION RESULTS
  MCQ        :   29 /   38  (76.32%)
  Free-form  :   32 /   62  (51.61%)
  Overall    :   61 /  100  (61.00%)
Variant: answer_focused
EVALUATION RESULTS
  MCQ        :   26 /   38  (68.42%)
  Free-form  :   28 /   62  (45.16%)
  Overall    :   54 /  100  (54.00%)
Variant: competition_style
EVALUATION RESULTS
  MCQ        :   27 /   38  (71.05%)
  Free-form  :   29 /   62  (46.77%)
  Overall    :   56 /  100  (56.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [33]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results['minimal_reasoning']:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 5 records to /content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition/results/prompt_engineering.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!